# ML Optimization - XGB

# 0.0 Importing Standard Libraries

In [ ]:
# Standard libraries

import sys, math, itertools, warnings, importlib, textwrap, random, ast, re, gc, pickle, json, os, sklearn, xgboost, joblib
from pathlib import Path

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"
NOTEBOOK_UTILS_SRC = PROJECT_ROOT / "notebook_utils" / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

if str(NOTEBOOK_UTILS_SRC) not in sys.path:
    sys.path.append(str(NOTEBOOK_UTILS_SRC))

from paths import DATA_RAW, DATA_INTERMEDIATE, FIGURES, MODELS, OUTPUTS, TABLES

import numpy as np
import pandas as pd
import operator
from datetime import datetime, date
from IPython.display import display, HTML
import pyfolio as pf
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import t, norm, entropy
from scipy.optimize import minimize
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split, TimeSeriesSplit, StratifiedKFold
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import xgboost as xgb

warnings.filterwarnings('ignore')

# 3.7.16 specific
from typing import List

In [ ]:
# Panda display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.max_colwidth', None) # Show all content of each column
pd.set_option('display.width', 1000)        # Set the display width to 1000 characters
pd.options.display.float_format = '{:,.5f}'.format
np.set_printoptions(precision=5, suppress=True)

print("pandas:", pd.__version__)
print("numpy :", np.__version__)
print("sklearn:", sklearn.__version__)
print("xgb:", xgboost.__version__)

### - Lib Import

In [ ]:
# My Functions: same directory
model = 'regime_model'
model_round = 'round_1'

model_input_path = DATA_RAW / "xls" / "input" / model
model_round_input_path = model_input_path / model_round
model_intermediate_path = DATA_INTERMEDIATE / model / model_round
model_table_output_path = TABLES / model / model_round  # REVIEW: confirm local target path for prior xls/output exports
model_csv_output_path = model_table_output_path / "csv"
model_xls_output_path = model_table_output_path / "xls"
model_figure_output_path = FIGURES / model / model_round
model_ml_output_path = MODELS / model / model_round
research_liquidity_path = DATA_RAW / "research" / "liquidity" / "xls"
research_fear_greed_path = DATA_RAW / "research" / "fear_greed" / "csv"


# 0.1 Selecting Sample Features

In [ ]:
GROUP_ALWAYS = [
    "atr_250",
    "atr_ema_a",
    "atr_ema_b",
    "atr_ktg",
    "atr_sma_a",
    "atr_sma_b",
    "beta",
    "cmf",
    "corr_1y",
    "corr_20d",
    "d_atr",
    "d_avol5",
    "d_avol50",
    "d_natr",
    "d_natr_ktg",
    "d_rsi",
    "kalmar_q",
    "pct_chg_open",
    "rsi",
    "rvol",
    "vol_20d",
    "vol_5d",
    "vol_60d",
    "spy_atr",
    "spy_rvol",
    # PCA / macro regime variables
    "PCA_Index_ma5",
    "PCA_ScaledIndex_ma5",
    "PCA_Index_ma20",
    "PCA_ScaledIndex_ma20",
    "PCA_Index_ma50",
    "PCA_ScaledIndex_ma50",
    "PCA_Raw_full",
    "PCA_Index_full",
    "fear_greed",
]

GROUP_CALENDAR = [
    "entry_hr_dec",
    "entry_hr_dec_to_close",
    "week_day_sin",
    "week_day_cos",
    "month_sin",
    "month_cos",
    "year_day_sin",
    "year_day_cos",
]

EXCLUDE_COLUMNS = {
    "mtm_pl",
    "pl_g",
    "pl_n",
    "fees",
    "Capital",
    "wins",
    "target_return",
    "target_down",
    "source_file",
    "entry_time",
    "exit_time",
    "normed_date",
    "symbol",
}

FEATURE_RULES = {
    "distance": lambda c: c.startswith("dist_"),
    "percent": lambda c: c.startswith("pct_"),
    "return": lambda c: c.startswith("ret_"),
    "dummy": lambda c: c.startswith("dumm_"),
    "volume_ratio": lambda c: ("vol" in c and "_rat" in c),
}

MANUAL_INCLUDE = []
MANUAL_EXCLUDE = []

def build_feature_list_from_columns(df):
    cols = df.columns.tolist()
    selected = []

    for col in GROUP_ALWAYS:
        if col in cols:
            selected.append(col)
    
    for col in GROUP_CALENDAR:
        if col in cols:
            selected.append(col)

    for _, rule in FEATURE_RULES.items():
        selected.extend([col for col in cols if rule(col)])

    selected.extend([col for col in MANUAL_INCLUDE if col in cols])

    selected = list(dict.fromkeys(selected))
    selected = [col for col in selected if col not in EXCLUDE_COLUMNS]
    selected = [col for col in selected if col not in MANUAL_EXCLUDE]

    return selected

# 1.0 Data Import

In [ ]:
names = [
    "partition_ins_80",
    "partition_ins_20",
    "partition_oos",
    "partition_ins",
]

# Option A: load into a dict
loaded = {
    name: pd.read_parquet(model_intermediate_path / f"{name}.parquet", engine="pyarrow")
    for name in names
}

partition_ins_80 = loaded["partition_ins_80"]
partition_ins_20 = loaded["partition_ins_20"]
partition_oos    = loaded["partition_oos"]
partition_ins    = loaded["partition_ins"]

print({k: v.shape for k, v in loaded.items()})

In [ ]:
feature_columns = build_feature_list_from_columns(partition_ins_80)
print(f'variables:{len(feature_columns)}')
print(textwrap.fill(", ".join(feature_columns), width = 250))

# 2.0 Correlation analysis

### - Reloading features selected for optimization

In [ ]:

optmz_list_all = feature_columns

### - Correlations with P/L

In [ ]:
corr_list = ['mtm_pl'] + optmz_list_all
correlation_matrix = partition_ins_80[corr_list].corr()

corr_mtmpl = (
    correlation_matrix[['mtm_pl']]
    .assign(mtm_pl_abs=lambda df: df['mtm_pl'].abs())
    .sort_values(by='mtm_pl_abs', ascending=False)
)

top_corr_mtmpl = corr_mtmpl[1:51]
top_corr_mtmpl_list = top_corr_mtmpl.index.tolist()
print(len(top_corr_mtmpl_list))
print(textwrap.fill(", ".join(top_corr_mtmpl_list), width = 250))


### - Highest correlated variables among themselves

In [ ]:
# Note for Codex: make ref_corr a parameter that can be changed in future runs
ref_corr = 0.85

high_corr_counts = (correlation_matrix > ref_corr).sum(axis=0) - 1  # subtract 1 to exclude correlation of the column with itself
filtered_high_corr_counts = high_corr_counts[high_corr_counts > 3].sort_values(ascending=False)
high_corr_list = filtered_high_corr_counts.index.tolist()
print(len(high_corr_list))
print(textwrap.fill(", ".join(high_corr_list), width = 250))


### - High correlation between variables highly correlated with P/L

In [ ]:
# High correlation between variables highly correlated with P/L
top_correlation_matrix = partition_ins_80[top_corr_mtmpl_list].corr()

ref_corr = 0.85
top_high_corr_counts = (top_correlation_matrix > ref_corr).sum(axis=0) - 1  # subtract 1 to exclude correlation of the column with itself
filtered_top_high_corr_counts = top_high_corr_counts[top_high_corr_counts > 3].sort_values(ascending=False)

top_high_corr_list = filtered_top_high_corr_counts.index.tolist()
print(len(top_high_corr_list))
print(textwrap.fill(", ".join(top_high_corr_list), width = 250))


### - Top 25 variables (high corr with P/L) and cross-correlations > 80%

In [ ]:
# Note for Codex: make cross_corr a parameter that can be changed in future runs
cross_corr = 0.85
top_correlation_matrix['consolidated'] = [
    [col for col in top_correlation_matrix.columns if top_correlation_matrix.loc[row, col] > cross_corr]
    for row in top_correlation_matrix.index
]

# Filter out rows where "consolidated" is empty
result_df = top_correlation_matrix[['consolidated']].loc[~top_correlation_matrix['consolidated'].apply(lambda x: len(x) == 0)]

# Cleaning data to remove repeated index names on "consolidated"
result_df['consolidated'] = [
    [item for item in row if str(index_name) not in str(item)]
    for index_name, row in zip(result_df.index, result_df['consolidated'])
]

print(result_df)


# 2.1 Feature Selection

### - Approach 1: Correlation to P/L

In [ ]:
variance_filter = partition_ins_80[optmz_list_all].var()
optmz_list_all = variance_filter[variance_filter > 1e-6].index.tolist()

# Step 1 — rank by correlation with P/L
corr_with_pl = (
    partition_ins_80[optmz_list_all]
    # .corrwith(partition_ins_80['wins'])
    .corrwith(partition_ins_80["mtm_pl"])
    .abs()
    .sort_values(ascending=False)
)

top_features = corr_with_pl.head(50).index.tolist()

# Step 2 — correlation matrix among top features
corr_matrix = partition_ins_80[top_features].corr().abs()

ref_corr = 0.85

corr_mtm_list = []
dropped_features = set()

for col in top_features:

    if col in dropped_features:
        continue

    corr_mtm_list.append(col)

    correlated = corr_matrix.index[corr_matrix[col] > ref_corr].tolist()

    for feat in correlated:
        if feat != col:
            dropped_features.add(feat)

print("Final feature count:", len(corr_mtm_list))
print(corr_mtm_list)

###  - Approach 2: Correlation to wins

In [ ]:
# Step 1 — rank by correlation with P/L
corr_with_pl = (
    partition_ins_80[optmz_list_all]
    .corrwith(partition_ins_80['wins'])
    .abs()
    .sort_values(ascending=False)
)

top_features = corr_with_pl.head(50).index.tolist()

# Step 2 — correlation matrix among top features
corr_matrix = partition_ins_80[top_features].corr().abs()

ref_corr = 0.85

corr_wins_list = []
dropped_features = set()

for col in top_features:

    if col in dropped_features:
        continue

    corr_wins_list.append(col)

    correlated = corr_matrix.index[corr_matrix[col] > ref_corr].tolist()

    for feat in correlated:
        if feat != col:
            dropped_features.add(feat)

print("Final feature count:", len(corr_wins_list))
print(corr_wins_list)

### - Approach 3: IC

In [ ]:
def compute_rank_ic_stats(df, features, target_col="wins", date_col="normed_date"):

    grouped = df.groupby(date_col)

    stats = []

    for feature in features:

        daily_ic = grouped.apply(
            lambda g: g[feature].corr(g[target_col], method="spearman")
        )

        stats.append({
            "feature": feature,
            "IC_mean": daily_ic.mean(),
            "IC_std": daily_ic.std(),
            "IC_IR": daily_ic.mean() / daily_ic.std() if daily_ic.std() > 0 else np.nan
        })

    return pd.DataFrame(stats).sort_values("IC_mean", ascending=False)


def prune_correlated_features(
    df: pd.DataFrame,
    ranked_features: list,
    corr_threshold: float = 0.85,
    method: str = "pearson",
):
    """
    Keep the highest-ranked feature from each highly correlated cluster.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing the candidate feature columns.
    ranked_features : list
        Features ordered from best to worst by some ranking metric
        (e.g. IC_IR, IC_mean, corr with wins, consensus vote, etc.).
    corr_threshold : float
        Absolute correlation threshold above which two features are treated
        as belonging to the same cluster.
    method : str
        Correlation method passed to DataFrame.corr(), typically
        'pearson' or 'spearman'.

    Returns
    -------
    selected_features : list
        Final pruned feature list.
    dropped_map : dict
        Dictionary mapping each kept feature to the list of dropped features
        that were removed because they were too correlated with it.
    corr_matrix : pd.DataFrame
        Absolute correlation matrix for the candidate ranked features.
    """

    # Keep only features that actually exist in df
    ranked_features = [f for f in ranked_features if f in df.columns]

    if not ranked_features:
        return [], {}, pd.DataFrame()

    # Correlation matrix on ranked features only
    corr_matrix = df[ranked_features].corr(method=method).abs()

    selected_features = []
    dropped_features = set()
    dropped_map = {}

    for feature in ranked_features:

        if feature in dropped_features:
            continue

        selected_features.append(feature)

        correlated_cluster = corr_matrix.index[
            corr_matrix.loc[feature] > corr_threshold
        ].tolist()

        # Drop all correlated features except itself
        to_drop = [f for f in correlated_cluster if f != feature]

        dropped_map[feature] = to_drop

        for f in to_drop:
            dropped_features.add(f)

    return selected_features, dropped_map, corr_matrix


def prune_correlated_features_with_summary(
    df: pd.DataFrame,
    ranked_features: list,
    corr_threshold: float = 0.85,
    method: str = "pearson",
):
    selected_features, dropped_map, corr_matrix = prune_correlated_features(
        df=df,
        ranked_features=ranked_features,
        corr_threshold=corr_threshold,
        method=method,
    )

    rows = []
    for keeper, dropped in dropped_map.items():
        rows.append({
            "keeper": keeper,
            "n_dropped": len(dropped),
            "dropped_features": ", ".join(dropped) if dropped else ""
        })

    summary_df = pd.DataFrame(rows).sort_values(
        ["n_dropped", "keeper"], ascending=[False, True]
    )

    return selected_features, dropped_map, corr_matrix, summary_df

In [ ]:
ic_table = compute_rank_ic_stats(partition_ins_80, optmz_list_all)

ic_top_list = (
    ic_table
    .sort_values("IC_IR", ascending=False)   # or "IC_mean"
    .head(50)
    ["feature"]
    .tolist()
)

print(len(ic_top_list))
print(textwrap.fill(", ".join(ic_top_list), width=250))

In [ ]:

ic_pruned_list, dropped_map, corr_matrix, cluster_summary = (
    prune_correlated_features_with_summary(
        df=partition_ins_80,
        ranked_features=ic_top_list,
        corr_threshold=0.85,
        method="pearson",
    )
)

print("Original top list:", len(ic_top_list))
print("Pruned list:", len(ic_pruned_list))
print(textwrap.fill(", ".join(ic_pruned_list), width=250))
cluster_summary.head(20)

### - Feature consolidation and pruning

In [ ]:
from collections import Counter

all_features = (corr_mtm_list + corr_wins_list + ic_pruned_list)
feature_votes = Counter(all_features)

selected_features = [
    feat for feat, votes in feature_votes.items()
    if votes >= 2
]

print(len(all_features))
print(textwrap.fill(", ".join(all_features), width=250))

print(len(selected_features))
print(textwrap.fill(", ".join(selected_features), width=250))

# 3.0 Pre ML-Optimization

### - Lib import

In [ ]:
import notebook_utils._dashboard_functions_one_symbol_v2 as _dashboard_functions_one_symbol_v2
importlib.reload(_dashboard_functions_one_symbol_v2)
from notebook_utils._dashboard_functions_one_symbol_v2 import dashboard

import notebook_utils._optimization_functions_37_v2 as _optimization_functions_37_v2
importlib.reload(_optimization_functions_37_v2)
from notebook_utils._optimization_functions_37_v2 import plot_features_vs_cvscore_rfecv_020

### - Dashboard for raw INS (cleaned) data

In [ ]:
## Creating Dashboard
## Creating stratification table (strat_ins_001)
# Unoptimized data (raw) of insample (ins)
LONG_SHORT = -1
ENTRY_FEE =  3.5

# Runding dashboard function
strat_ins_001, ins_bydate_001 = dashboard(partition_ins_80, ENTRY_FEE, 'ins_raw', LONG_SHORT, 'short', 'max')


# 4.0 ML Optimization

### - Missing observations

In [ ]:
# Infis first
numeric_df = partition_ins_80.select_dtypes(include=[np.number])
inf_cols = numeric_df.columns[np.isinf(numeric_df).any()]
print("Columns with Inf/-Inf values:", inf_cols.tolist())
# NaN second
nan_cols = partition_ins_80.columns[partition_ins_80.isna().any()]
print("Columns with NaN values:", nan_cols.tolist())

# Summary
_df_ = partition_ins_80
# NaN counts (all columns)
nan_count = _df_.isna().sum()
nan_count = nan_count[nan_count > 0]

# Inf/-Inf counts (numeric columns only)
numeric_df = _df_.select_dtypes(include=[np.number])
inf_count = np.isinf(numeric_df).sum()
inf_count = inf_count[inf_count > 0]

# Combine into one summary DataFrame
summary_df = pd.DataFrame({
    'NaN Count': nan_count,
    'Inf Count': inf_count
}).fillna(0).astype(int)

print(summary_df)

### - ML Train/Test splits

In [ ]:
# Two options: all_features or selected_features

USE_SELECTED_FEATURES = True
ml_list = selected_features if USE_SELECTED_FEATURES else all_features

print(len(ml_list))

In [ ]:
X_train = partition_ins_80[ml_list]
y_train  = partition_ins_80['wins']

X_test = partition_ins_20[ml_list]
y_test  = partition_ins_20['wins']

# Only dependent variable needed for OOS as we will fit the optimal number of features of X, which is calculated later
# partition_oos

y_oos  = partition_oos['wins']

print(f'train: {len(y_train)}')
print("Train Win %", np.mean(y_train))
print("")
print(f'test: {len(y_test)}')
print("Test Win %", np.mean(y_test))
print("")
print(f'test: {len(y_oos)}')
print("Test Win %", np.mean(y_oos))


### - Hyperparameter tunning (*No need to run every time)

In [ ]:
import notebook_utils._grid_search_37_v2 as _grid_search_37_v2
importlib.reload(_grid_search_37_v2)
from notebook_utils._grid_search_37_v2 import (
    build_base_xgb, 
    param_distributions, 
    randomized_search_on_sample, 
    final_refit_with_early_stopping, 
    tune_xgb_fast_compatible
)

#### - A1: Faster random grid search with full data and selected (high corr) features

In [ ]:

model, best = tune_xgb_fast_compatible(X_train, y_train, X_test, y_test, search_rows=1_000, n_iter=60, cv_splits=5)
print("Best params:", best)
print("Best ntree limit:", getattr(model, "best_ntree_limit", None))
print("Best iteration:", getattr(model, "best_iteration", None))


#### - B1: Full grid search with full data and selected (high corr) features

In [ ]:
xgb_tunning = xgb.XGBClassifier(learning_rate=0.1, random_state=42, eval_metric='logloss', use_label_encoder=False)

param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20],      
    'gamma': [0.1, 0.2, 0.3], 
    'subsample': [0.7, 0.8, 0.9],  
    'colsample_bytree': [0.7, 0.8, 0.9], 
    'reg_alpha': [0.01, 0.1, 1],  
    }

grid_search = GridSearchCV(estimator=xgb_tunning, param_grid=param_grid, cv=5, scoring='precision', refit=True, verbose=2, n_jobs=-1)
grid_search.fit(X_train, y_train)

# Step 5: Evaluate the model
best_xgb = grid_search.best_estimator_  # Get the best model
y_pred = best_xgb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print("Best Parameters:", grid_search.best_params_)
print("Test Set Accuracy:", accuracy)


In [ ]:

best_params = grid_search.best_params_

# Step 1: Create a new XGB with the best parameters
best_xgb = xgb.XGBClassifier(**best_params, learning_rate=0.1, random_state=42, eval_metric='logloss', objective="binary:logistic", use_label_encoder=False, tree_method="hist", n_jobs=-1)

# Step 2: Train the model on your training data
best_xgb.fit(X_train, y_train)

# Step 3: Evaluate or use the model
pred_train = best_xgb.predict(X_train)
pred_test = best_xgb.predict(X_test)

print(classification_report(y_train,pred_train))
print(confusion_matrix(y_train,pred_train))

print(classification_report(y_test,pred_test))
print(confusion_matrix(y_test,pred_test))


In [ ]:
print("ML Predictions from Train data")
print("Obs:", len(pred_train))
print("Mean:", np.mean(pred_train))
print("P/L+", np.sum(pred_train == 1))

print("")
print("ML Predictions from Test data")
print("Obs:", len(pred_test))
print("Mean:", np.mean(pred_test))
print("P/L+", np.sum(pred_test == 1))


### - Feature optimization (* No need to run every time)

In [ ]:

X_train_indexed = partition_ins_80.set_index('normed_date') 
daily_counts = X_train_indexed.resample('1D').size()

# Filter out days with 0 trades (weekends/holidays) to get a true average
active_days = daily_counts[daily_counts > 0]

avg_rows_per_day = active_days.mean()
median_rows_per_day = active_days.median()

print(f"Average rows per day: {int(avg_rows_per_day)}")
print(f"Median rows per day:  {int(median_rows_per_day)}")

# 2. Define your desired "Window" in time
# For event-driven strategies, 3-6 months is common for regime adaptation.
target_lookback_months = 9 
target_lookback_days = target_lookback_months * 21  # ~21 trading days/month

# 3. Calculate max_train_size
calculated_size = int(avg_rows_per_day * target_lookback_days)

print(f"Recommended max_train_size: {calculated_size}")

In [ ]:

MAX_TRAIN_SIZE = calculated_size
cv_rolling = TimeSeriesSplit(max_train_size=MAX_TRAIN_SIZE, n_splits=5)

# Setting finner steps (to 1 from 10)
rfecv = RFECV(estimator=best_xgb, step=1, min_features_to_select=5, cv=cv_rolling, scoring='precision', n_jobs=-1)

rfecv.fit(X_train, y_train)

# Number of selected features
print(f"Optimal number of features: {rfecv.n_features_}")
# print(f"Optimal features: {list(rfecv.get_feature_names_out())}")

selected_mask = rfecv.support_
selected_features = X_train.columns[selected_mask]
print(f"Optimal features: {list(selected_features)}")


In [ ]:
plot_features_vs_cvscore_rfecv_020(rfecv, X_train, scoring_label="recall", increasing_x=True)

In [ ]:
selected_features = X_train.columns[rfecv.support_]
importances = rfecv.estimator_.feature_importances_
pd.Series(importances, index=selected_features).sort_values(ascending=False)


In [ ]:
scores = rfecv.grid_scores_

# 2. Calculate the number of features for each score
# We start from min_features and add the step size for each subsequent score
n_features = range(
    rfecv.min_features_to_select, 
    rfecv.min_features_to_select + (len(scores) * rfecv.step), 
    rfecv.step
)

# Note: If len(n_features) doesn't match len(scores) due to edge cases in older versions, 
# you can simply use: range(1, len(scores) + 1) for a rough view.

# 3. Plot
plt.figure(figsize=(10, 6))
plt.xlabel("Number of features selected")
plt.ylabel("Cross validation score (Precision)")
plt.plot(range(1, len(scores) + 1), scores) # Simplified X-axis for safety
plt.title("Recursive Feature Elimination Results")
plt.show()

# Print the scores to find the "knee"
# for i, score in enumerate(scores):
#     print(f"Step {i+1}: Score {score:.4f}")

In [ ]:
n_features_list = range(
    rfecv.min_features_to_select, 
    rfecv.min_features_to_select + (len(scores) * rfecv.step), 
    rfecv.step
)

# 2. Define your "Complexity Tolerance"
# In trading, 0.01 (1%) is a good starting point. 
# You are willing to lose 1% accuracy to drop features.
tolerance = 0.01 

# 3. Find the max score
max_score = np.max(scores)
threshold = max_score * (1 - tolerance)

# 4. Find the smallest number of features that meets the threshold
# We zip them, filter by score >= threshold, and take the first (smallest) one.
parsimonious_set = next(
    (n, s) for n, s in zip(n_features_list, scores) 
    if s >= threshold
)

print(f"Absolute Max Score: {max_score:.4f}")
print(f"Acceptable Threshold: {threshold:.4f} (Tolerance: {tolerance})")
print(f"Parsimonious Choice: {parsimonious_set[0]} features (Score: {parsimonious_set[1]:.4f})")

### - Running best specification (per RFECV)

In [ ]:
rfecv_vars =  selected_features

X_train_rfecv = partition_ins_80[rfecv_vars]
X_test_rfecv = partition_ins_20[rfecv_vars]
X_oos_rfecv = partition_oos[rfecv_vars]


In [ ]:

# Fitting/evaluating: INS Train
best_xgb.fit(X_train_rfecv, y_train)

# Predicting/evaluating: INS Train
pred_train_rfecv = best_xgb.predict(X_train_rfecv)
yhat_train = best_xgb.predict_proba(X_train_rfecv)
yhat_train_1 = yhat_train[:, 1]  # Probabilities of class 1
print("Obs:", len(yhat_train_1))
print("Mean:", np.mean(yhat_train_1))

# Predicting/evaluating: INS Test
pred_test_rfecv = best_xgb.predict(X_test_rfecv)
yhat_test = best_xgb.predict_proba(X_test_rfecv)
yhat_test_1 = yhat_test[:, 1]  # Probabilities of class 1
print("Obs:", len(yhat_test_1))
print("Mean:", np.mean(yhat_test_1))

# Predicting/evaluating: OOS
pred_oos_rfecv = best_xgb.predict(X_oos_rfecv)
yhat_oos = best_xgb.predict_proba(X_oos_rfecv)
yhat_oos_1 = yhat_oos[:, 1]  # Probabilities of class 1
print("Obs:", len(yhat_oos_1))
print("Mean:", np.mean(yhat_oos_1))


In [ ]:
print(classification_report(y_train,pred_train_rfecv))
print(confusion_matrix(y_train,pred_train_rfecv))

print(classification_report(y_test,pred_test_rfecv))
print(confusion_matrix(y_test,pred_test_rfecv))

print(classification_report(y_oos,pred_oos_rfecv))
print(confusion_matrix(y_oos,pred_oos_rfecv))


### - ROC Curve

In [ ]:
# ROC for Train
fpr_train, tpr_train, _ = roc_curve(y_train, yhat_train_1)
roc_auc_train = auc(fpr_train, tpr_train)

# ROC for Test
fpr_test, tpr_test, _ = roc_curve(y_test, yhat_test_1)
roc_auc_test = auc(fpr_test, tpr_test)

# Plot both
plt.figure(figsize=(6, 5))
plt.plot(fpr_train, tpr_train, label=f'Train ROC (AUC = {roc_auc_train:.2f})')
plt.plot(fpr_test, tpr_test, label=f'Test ROC (AUC = {roc_auc_test:.2f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='grey')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - XGBoost (Train vs Test)')
plt.legend(loc='lower right')
plt.grid(True)
plt.tight_layout()
plt.show()

### - Saving specification for KITE load* (saving only once)

In [ ]:
model = 'regime_model'
model_round = 'round_1'
direction = "long" if LONG_SHORT == 1 else "short"
filename = f"xgb_{model}_{direction}.pkl"

model_ml_output_path.mkdir(parents=True, exist_ok=True)
ml_path = model_ml_output_path / filename

with open(ml_path, "wb") as f:
    pickle.dump(best_xgb, f)

print(f"Model saved at: {ml_path}")

In [ ]:
# Just to confirm list of features
print(best_xgb.get_booster().feature_names)

### - Data copy 3

In [ ]:
partition_ins_80_001 = partition_ins_80.copy()
partition_ins_20_001 = partition_ins_20.copy()
partition_oos_001 = partition_oos.copy()

### - Merging predicted values to INS-80%/20% and OOS

In [ ]:
ML_PROBA_COL = "ml_proba_1"

prediction_map = {
    "partition_ins_80_001": (partition_ins_80_001, yhat_train_1),
    "partition_ins_20_001": (partition_ins_20_001, yhat_test_1),
    "partition_oos_001": (partition_oos_001, yhat_oos_1),
}

for df_name, (df, yhat) in prediction_map.items():
    if len(df) != len(yhat):
        raise ValueError(f"Length mismatch between {df_name} and prediction array.")

    df[ML_PROBA_COL] = yhat
    df.reset_index(drop=True, inplace=True)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5))
sns.scatterplot(x='ml_proba_1', y='mtm_pl', data=partition_ins_80_001, ax=axes[0])
sns.scatterplot(x='entry_hr_dec', y='mtm_pl', data=partition_ins_80_001, ax=axes[1])
sns.scatterplot(x='fear_greed', y='mtm_pl', data=partition_ins_80_001, ax=axes[2])
sns.scatterplot(x='PCA_Index_full', y='ml_proba_1', data=partition_ins_80_001, ax=axes[3])


### - Lib import

In [ ]:
import notebook_utils._data_explore_functions as _data_explore_functions
importlib.reload(_data_explore_functions)
from notebook_utils._data_explore_functions import append_summary, reset_summary

In [ ]:
reset_summary()
summary_df = append_summary(partition_ins_80_001, "partition_ins_80_001")
summary_df = append_summary(partition_ins_20_001, "partition_ins_20_001")
summary_df = append_summary(partition_oos_001, "partition_oos_001")
print(summary_df)


#### - Saving pre-optimized data (with all features) and ML prob

In [ ]:
DEBUG_ROWS = 50000  # or 50000 for quick iteration

dfs = {
    "partition_ins_80_001": partition_ins_80_001,
    "partition_ins_20_001": partition_ins_20_001,
    "partition_oos_001":    partition_oos_001,
}

for name, df in dfs.items():
    df_out = df if DEBUG_ROWS is None else df.head(DEBUG_ROWS)

    rows_written = len(df_out)
    print(f"{name}: writing {rows_written:,} rows")

    df.to_parquet(model_intermediate_path / f"{name}.parquet",
                  engine="pyarrow",   # if missing, use engine="fastparquet"
                  index=False,
                  compression="zstd") # if unavailable, try "snappy"
